####Requirement
1. Read raw data from flight_time_raw table
2. Apply transformations to time values as hour to minute interval

    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

In [0]:
%sql

select * from dev.spark_db.flight_time_raw limit  10

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946
2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946
2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946
2000-01-01,DL,1997,BOS,"Boston, MA",ATL,"Atlanta, GA",1715,1720,1955,10,2013,2005,0,946
2000-01-01,DL,2065,BOS,"Boston, MA",ATL,"Atlanta, GA",2015,2010,2230,10,2300,2240,0,946
2000-01-01,US,2619,BOS,"Boston, MA",ATL,"Atlanta, GA",650,649,956,7,955,1003,0,946
2000-01-01,US,2621,BOS,"Boston, MA",ATL,"Atlanta, GA",1440,1446,1713,4,1738,1717,0,946
2000-01-01,DL,346,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",1740,1744,1957,9,2008,2006,0,449
2000-01-01,DL,412,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",1345,1345,1552,9,1622,1601,0,449
2000-01-01,DL,299,BUF,"Buffalo, NY",ATL,"Atlanta, GA",1245,1245,1443,5,1455,1448,0,712


read flights data from datalake into a table. step by step transformatted into analysis. Convert the above 5 fields into required format. 

Below are the steps

#### STEP1. Read data to create a dataframe

In [0]:
flight_time_raw_df = spark.read.table("dev.spark_db.flight_time_raw")

####2. Develop logic to transform CRS_DEP_TIME to an interval

The logic is to make the CRS_DEP_TIME into 4 characters long. if it has 3 characters long then add zero before the digit. example- 700 will become 0700. 
And take 2 digits from 4 characters to call it HH as Hour

In [0]:
from pyspark.sql.functions import expr

step_1_df = (
    flight_time_raw_df.withColumns({
    "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    })
)

step_2_df = (
    step_1_df.withColumns({
        "CRS_DEP_TIME_NEW": expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)")
    })
)

In [0]:
step_2_df.limit(2).display()

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE,CRS_DEP_TIME_HH,CRS_DEP_TIME_MM,CRS_DEP_TIME_NEW
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946,11,15,INTERVAL '11:15' HOUR TO MINUTE
2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946,13,15,INTERVAL '13:15' HOUR TO MINUTE


####3. Develop a reusable function

In [0]:
 def get_interval(hhmm_value):
     from pyspark.sql.functions import expr

     return expr(f"""
                 cast(concat(left(lpad({hhmm_value}, 4, '0'), 2), ':', 
                             right(lpad({hhmm_value}, 4, '0'), 2)) 
                             AS INTERVAL HOUR TO MINUTE)
                 """)

####4. Apply function to dataframe

In [0]:
result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
        "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

####5. Save results to the table 

In [0]:
result_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time")

In [0]:
%sql

select * from dev.spark_db.flight_time limit 10

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '11:15' HOUR TO MINUTE,INTERVAL '11:13' HOUR TO MINUTE,INTERVAL '13:43' HOUR TO MINUTE,INTERVAL '05' MINUTE,INTERVAL '14:00' HOUR TO MINUTE,INTERVAL '13:48' HOUR TO MINUTE,0,946
2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '13:15' HOUR TO MINUTE,INTERVAL '13:11' HOUR TO MINUTE,INTERVAL '15:36' HOUR TO MINUTE,INTERVAL '07' MINUTE,INTERVAL '15:59' HOUR TO MINUTE,INTERVAL '15:43' HOUR TO MINUTE,0,946
2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '14:15' HOUR TO MINUTE,INTERVAL '14:14' HOUR TO MINUTE,INTERVAL '16:42' HOUR TO MINUTE,INTERVAL '09' MINUTE,INTERVAL '17:21' HOUR TO MINUTE,INTERVAL '16:51' HOUR TO MINUTE,0,946
2000-01-01,DL,1997,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '17:15' HOUR TO MINUTE,INTERVAL '17:20' HOUR TO MINUTE,INTERVAL '19:55' HOUR TO MINUTE,INTERVAL '10' MINUTE,INTERVAL '20:13' HOUR TO MINUTE,INTERVAL '20:05' HOUR TO MINUTE,0,946
2000-01-01,DL,2065,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:10' HOUR TO MINUTE,INTERVAL '22:30' HOUR TO MINUTE,INTERVAL '10' MINUTE,INTERVAL '23:00' HOUR TO MINUTE,INTERVAL '22:40' HOUR TO MINUTE,0,946
2000-01-01,US,2619,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '06:50' HOUR TO MINUTE,INTERVAL '06:49' HOUR TO MINUTE,INTERVAL '09:56' HOUR TO MINUTE,INTERVAL '07' MINUTE,INTERVAL '09:55' HOUR TO MINUTE,INTERVAL '10:03' HOUR TO MINUTE,0,946
2000-01-01,US,2621,BOS,"Boston, MA",ATL,"Atlanta, GA",INTERVAL '14:40' HOUR TO MINUTE,INTERVAL '14:46' HOUR TO MINUTE,INTERVAL '17:13' HOUR TO MINUTE,INTERVAL '04' MINUTE,INTERVAL '17:38' HOUR TO MINUTE,INTERVAL '17:17' HOUR TO MINUTE,0,946
2000-01-01,DL,346,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",INTERVAL '17:40' HOUR TO MINUTE,INTERVAL '17:44' HOUR TO MINUTE,INTERVAL '19:57' HOUR TO MINUTE,INTERVAL '09' MINUTE,INTERVAL '20:08' HOUR TO MINUTE,INTERVAL '20:06' HOUR TO MINUTE,0,449
2000-01-01,DL,412,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",INTERVAL '13:45' HOUR TO MINUTE,INTERVAL '13:45' HOUR TO MINUTE,INTERVAL '15:52' HOUR TO MINUTE,INTERVAL '09' MINUTE,INTERVAL '16:22' HOUR TO MINUTE,INTERVAL '16:01' HOUR TO MINUTE,0,449
2000-01-01,DL,299,BUF,"Buffalo, NY",ATL,"Atlanta, GA",INTERVAL '12:45' HOUR TO MINUTE,INTERVAL '12:45' HOUR TO MINUTE,INTERVAL '14:43' HOUR TO MINUTE,INTERVAL '05' MINUTE,INTERVAL '14:55' HOUR TO MINUTE,INTERVAL '14:48' HOUR TO MINUTE,0,712
